# Exploratory Data Analysis (EDA) & Data Curation
## Practical Notebook — Data Analysis Programming

**Dataset:** `weather_problems.json` — Australian daily weather observations (372 records with intentional data quality issues)

**Topics covered:**
1. Setup & Data Loading
2. EDA — Load & Profile Data
3. EDA — Univariate Analysis
4. EDA — Descriptive Analytics: Central Tendency & Dispersion
5. EDA — Quantiles, Skewness, and Kurtosis
6. EDA — Bivariate Analysis & Correlation
7. Data Curation — Data Quality Dimensions
8. Data Curation — Outlier Detection
9. Data Curation — Outlier Treatment
10. Data Curation — Missing Data & Imputation
11. Data Curation — Feature Engineering
12. Data Curation — Feature Selection

In [ ]:
import sys, subprocess

_pkgs = [
    "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "scipy",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet"] + _pkgs
)
print("All packages installed successfully.")

---
## 1. Setup & Data Loading

> **Dataset**: `weather_problems.json` — a version of the Australian weather dataset with intentional data quality problems introduced for teaching purposes.
>
> **Known issues embedded in the data:**
> - `Sunshine`, `WindGustSpeed`, `WindSpeed9am`, `Evaporation` stored as **strings** instead of numerics
> - ~10% **null values** in `Sunshine`, `Cloud9am`, `Cloud3pm`, `Humidity3pm`, `Evaporation`
> - **Extreme outliers**: `MaxTemp = 88°C`, `Pressure9am = 500 hPa`, `Humidity9am = 150%`, etc.
> - **Inconsistent casing** in `RainToday` / `RainTomorrow`: `"YES"`, `"yes"`, `"No"`, `"NO"`
> - **6 duplicate rows** appended at the end

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Matplotlib style
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
sns.set_theme(style="whitegrid", palette="muted")

# ── Load dataset ─────────────────────────────────────────────────────────────
DATA_PATH = Path("weather_problems.json")

with open(DATA_PATH) as f:
    records = json.load(f)

df_raw = pd.DataFrame(records)
print(f"Shape: {df_raw.shape}")
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
df_raw.head(3)

---
## 2. EDA — Load & Profile Data

The first EDA step is **profiling**: understand data types, null counts, unique values, and basic statistics.
This maps to `df.info()`, `df.describe()`, and `.isnull().sum()` from the slides.

In [ ]:
# ── df.info() equivalent ────────────────────────────────────────────────────
print("=" * 55)
print("DATASET INFO")
print("=" * 55)
df_raw.info()

print("\n" + "=" * 55)
print("NULL COUNTS PER COLUMN")
print("=" * 55)
null_counts = df_raw.isnull().sum()
null_pct    = (null_counts / len(df_raw) * 100).round(2)
null_report = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print(null_report[null_report["null_count"] > 0].to_string())

In [ ]:
# ── df.describe() ────────────────────────────────────────────────────────────
print("NUMERIC SUMMARY (df.describe())")
print("-" * 55)
# Only numeric columns for now (strings will be flagged)
df_raw.describe(include="number").round(2)

In [ ]:
# ── Detect type anomalies ────────────────────────────────────────────────────
print("COLUMNS STORED AS OBJECT (string) dtype:")
print("-" * 45)
for col in df_raw.select_dtypes(include="object").columns:
    sample = df_raw[col].dropna().head(5).tolist()
    print(f"  {col:20s} unique={df_raw[col].nunique():4d}  sample={sample}")

print()
print("DUPLICATE ROWS:", df_raw.duplicated().sum())

In [ ]:
# ── Null heatmap ─────────────────────────────────────────────────────────────
null_matrix = df_raw.isnull().astype(int)

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(null_matrix.T, aspect="auto", cmap="Reds", vmin=0, vmax=1)
ax.set_yticks(range(len(df_raw.columns)))
ax.set_yticklabels(df_raw.columns, fontsize=8)
ax.set_xlabel("Record index", fontsize=10)
ax.set_title("Missing Value Map (red = null)", fontsize=12)
plt.colorbar(im, ax=ax, label="1 = null")
plt.tight_layout()
plt.show()

print(f"\nTotal cells: {df_raw.size:,}  |  Total nulls: {df_raw.isnull().sum().sum()}  "
      f"({df_raw.isnull().sum().sum()/df_raw.size*100:.1f}%)")

---
## 3. EDA — Univariate Analysis

Examine each variable in isolation:
- **Continuous variables**: histograms + KDE to see distribution shape
- **Categorical variables**: bar charts to see class frequencies

From the EDA pipeline in the slides: *Step 2 — Univariate Analysis (histograms, box plots)*

In [ ]:
# ── Convert string columns to numeric for analysis ───────────────────────────
# (temporary — full cleaning happens in Section 7)
df_num = df_raw.copy()
for col in ["Sunshine", "WindGustSpeed", "WindSpeed9am", "Evaporation"]:
    df_num[col] = pd.to_numeric(df_num[col], errors="coerce")

NUMERIC_COLS = [
    "MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
    "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
    "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
    "Cloud9am", "Cloud3pm", "Temp9am", "Temp3pm", "RISK_MM",
]

# ── Histograms with KDE ────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 5, figsize=(18, 13))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    data = df_num[col].dropna()
    axes[i].hist(data, bins=30, color="#4c72b0", alpha=0.7, edgecolor="white", density=True)
    try:
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(data, bw_method="scott")
        xs  = np.linspace(data.min(), data.max(), 200)
        axes[i].plot(xs, kde(xs), color="#c44e52", lw=2)
    except Exception:
        pass
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")
    axes[i].tick_params(labelsize=7)

# Hide unused axes
for j in range(len(NUMERIC_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Univariate Analysis — Histograms + KDE", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Box plots — good for spotting outliers ───────────────────────────────────
key_cols = ["MinTemp", "MaxTemp", "Rainfall", "Humidity9am", "Humidity3pm",
            "Pressure9am", "Pressure3pm", "WindGustSpeed"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(key_cols):
    data = df_num[col].dropna()
    axes[i].boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor="#4c72b0", alpha=0.6),
                    medianprops=dict(color="#c44e52", lw=2),
                    flierprops=dict(marker="o", markersize=3, color="red", alpha=0.5))
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

plt.suptitle("Box Plots — Outliers are visible as red circles beyond the whiskers", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical columns ──────────────────────────────────────────────────────
cat_cols = ["WindGustDir", "WindDir9am", "WindDir3pm", "RainToday", "RainTomorrow"]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(cat_cols):
    counts = df_raw[col].value_counts()
    axes[i].bar(counts.index, counts.values, color="#55a868", edgecolor="white", alpha=0.85)
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(axis="x", rotation=60, labelsize=7)
    axes[i].tick_params(axis="y", labelsize=8)

plt.suptitle("Categorical Columns — Value Counts\n(notice mixed casing in RainToday / RainTomorrow)", fontsize=12)
plt.tight_layout()
plt.show()

print("\nRainToday unique values:", sorted(df_raw["RainToday"].dropna().unique()))
print("RainTomorrow unique values:", sorted(df_raw["RainTomorrow"].dropna().unique()))

---
## 4. EDA — Descriptive Analytics: Central Tendency & Dispersion

From the slides:
- **Central Tendency**: Mean (sensitive to outliers), Median (robust), Mode (categorical)
- **Dispersion**: Variance $s^2$, Std Dev $s$, IQR = $Q_3 - Q_1$, Range

The **mean vs median gap** is a first indicator of outliers or skewness.

In [ ]:
from scipy import stats as sp_stats

summary_rows = []
for col in NUMERIC_COLS:
    s = df_num[col].dropna()
    if len(s) == 0:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    summary_rows.append({
        "Column":    col,
        "Mean":      round(s.mean(), 2),
        "Median":    round(s.median(), 2),
        "Std Dev":   round(s.std(), 2),
        "Variance":  round(s.var(), 2),
        "IQR":       round(q3 - q1, 2),
        "Range":     round(s.max() - s.min(), 2),
        "Mean-Med Δ": round(abs(s.mean() - s.median()), 2),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Column")
print("DESCRIPTIVE STATISTICS")
print("=" * 70)
print(summary_df.to_string())

print("\n🔍 Large Mean–Median gaps suggest outliers or strong skew:")
big_gap = summary_df[summary_df["Mean-Med Δ"] > summary_df["Std Dev"] * 0.3]
if len(big_gap):
    print(big_gap[["Mean", "Median", "Mean-Med Δ", "Std Dev"]].to_string())
else:
    print("  (none)")

In [ ]:
# ── Mean vs Median bar comparison ─────────────────────────────────────────────
show_cols = ["MinTemp", "MaxTemp", "Rainfall", "Pressure9am", "Humidity9am", "WindGustSpeed"]
means     = [df_num[c].mean() for c in show_cols]
medians   = [df_num[c].median() for c in show_cols]

x = np.arange(len(show_cols))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - w/2, means,   w, label="Mean",   color="#4c72b0", alpha=0.85)
ax.bar(x + w/2, medians, w, label="Median", color="#55a868", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(show_cols, fontsize=10)
ax.set_ylabel("Value")
ax.set_title("Mean vs Median — Gap reveals outlier influence\n(large gap = outliers pulling the mean)", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. EDA — Quantiles, Skewness, and Kurtosis

From the slides:
- **Quantiles** partition the sorted data into equal groups ($Q_1$, $Q_2$, $Q_3$, percentiles)
- **Skewness** $\gamma_1 > 0$: right-skewed (long right tail); $\gamma_1 < 0$: left-skewed
- **Kurtosis** $\gamma_2 > 0$: leptokurtic (heavy tails); $\gamma_2 < 0$: platykurtic (light tails)

In [ ]:
shape_rows = []
for col in NUMERIC_COLS:
    s = df_num[col].dropna()
    if len(s) < 10:
        continue
    skew = round(sp_stats.skew(s), 3)
    kurt = round(sp_stats.kurtosis(s), 3)   # excess kurtosis (normal = 0)
    q1, q2, q3 = s.quantile(0.25), s.quantile(0.50), s.quantile(0.75)
    p95 = s.quantile(0.95)
    shape_rows.append({
        "Column": col,
        "Q1": round(q1, 2),
        "Q2 (Median)": round(q2, 2),
        "Q3": round(q3, 2),
        "P95": round(p95, 2),
        "Skewness": skew,
        "Kurtosis (excess)": kurt,
        "Shape": ("right-skewed" if skew > 0.5 else "left-skewed" if skew < -0.5 else "symmetric"),
    })

shape_df = pd.DataFrame(shape_rows).set_index("Column")
print("QUANTILES, SKEWNESS AND KURTOSIS")
print("=" * 70)
print(shape_df.to_string())

In [ ]:
# ── Distribution shape visualisation for selected columns ────────────────────
shape_cols = ["Rainfall", "MaxTemp", "Humidity3pm", "Pressure9am"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(shape_cols):
    s = df_num[col].dropna()
    axes[i].hist(s, bins=35, density=True, color="#4c72b0", alpha=0.65, edgecolor="white")
    # KDE
    try:
        kde = sp_stats.gaussian_kde(s)
        xs  = np.linspace(s.min(), s.max(), 200)
        axes[i].plot(xs, kde(xs), color="#c44e52", lw=2.5)
    except Exception:
        pass
    skew_val = sp_stats.skew(s)
    axes[i].axvline(s.mean(),   color="orange", lw=1.5, linestyle="--", label="mean")
    axes[i].axvline(s.median(), color="green",  lw=1.5, linestyle=":",  label="median")
    axes[i].set_title(f"{col}\nskewness={skew_val:.2f}", fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle("Distribution Shapes — skewness visible as mean vs median offset", fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. EDA — Bivariate Analysis & Correlation

From the slides:
- **Pearson** $r$: linear association, $r \in [-1, 1]$; sensitive to outliers
- **Spearman** $\rho$: rank-based, robust to outliers; monotonic relationships
- $|r| \geq 0.7$ → strong;  $0.4 \leq |r| < 0.7$ → moderate

>  **Correlation ≠ Causation**

In [ ]:
# ── Pearson & Spearman correlation heatmaps ───────────────────────────────────
corr_cols = ["MinTemp","MaxTemp","Rainfall","Humidity9am","Humidity3pm",
             "Pressure9am","Pressure3pm","Temp9am","Temp3pm",
             "WindGustSpeed","Sunshine","RISK_MM"]

df_corr = df_num[corr_cols].dropna()

pearson  = df_corr.corr(method="pearson")
spearman = df_corr.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, corr_mat, title in zip(axes,
                                [pearson, spearman],
                                ["Pearson Correlation ($r$)", "Spearman Correlation ($\\rho$)"]):
    mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
    sns.heatmap(corr_mat, ax=ax, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, vmin=-1, vmax=1, linewidths=0.4,
                annot_kws={"size": 7}, mask=mask)
    ax.set_title(title, fontsize=12)
    ax.tick_params(labelsize=8)

plt.suptitle("Correlation Analysis — Pearson vs Spearman\n"
             "(Spearman is more robust to outliers)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Strong correlations
print("\nStrong correlations |r| >= 0.7 (Pearson):")
for c1 in pearson.columns:
    for c2 in pearson.columns:
        if c1 >= c2:
            continue
        v = pearson.loc[c1, c2]
        if abs(v) >= 0.7:
            print(f"  {c1:15s} — {c2:15s}  r = {v:+.3f}")

In [ ]:
# ── Scatter plots for strongly correlated pairs ──────────────────────────────
strong_pairs = [("Temp9am", "Temp3pm"), ("Pressure9am", "Pressure3pm"),
                ("Humidity9am", "Humidity3pm"), ("MinTemp", "MaxTemp")]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (x_col, y_col) in enumerate(strong_pairs):
    subset = df_num[[x_col, y_col]].dropna()
    r, _   = sp_stats.pearsonr(subset[x_col], subset[y_col])
    axes[i].scatter(subset[x_col], subset[y_col], alpha=0.4, s=15, color="#4c72b0")
    # Regression line
    m, b = np.polyfit(subset[x_col], subset[y_col], 1)
    xs   = np.linspace(subset[x_col].min(), subset[x_col].max(), 100)
    axes[i].plot(xs, m*xs + b, color="#c44e52", lw=2)
    axes[i].set_xlabel(x_col, fontsize=9)
    axes[i].set_ylabel(y_col, fontsize=9)
    axes[i].set_title(f"r = {r:+.3f}", fontsize=10)

plt.suptitle("Scatter Plots — Strongly Correlated Variable Pairs", fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Data Curation — Data Quality Dimensions

From the slides, data quality has **7 dimensions**:
Accuracy, Completeness, Consistency, Timeliness, Uniqueness, Validity, Integrity.

Here we systematically identify every issue in `weather_problems.json` and document it before fixing it.

In [ ]:
issues = []

# 1. Completeness — null values
null_summary = df_raw.isnull().sum()
for col, cnt in null_summary[null_summary > 0].items():
    pct = cnt / len(df_raw) * 100
    issues.append({
        "Dimension":   "Completeness",
        "Column":      col,
        "Issue":       f"{cnt} null values ({pct:.1f}%)",
        "Severity":    "High" if pct > 10 else "Medium",
    })

# 2. Validity — type mismatches (numeric stored as string)
string_num_cols = ["Sunshine", "WindGustSpeed", "WindSpeed9am", "Evaporation"]
for col in string_num_cols:
    issues.append({
        "Dimension": "Validity",
        "Column":    col,
        "Issue":     f"Stored as string, expected numeric",
        "Severity":  "High",
    })

# 3. Consistency — mixed casing in categorical flags
for col in ["RainToday", "RainTomorrow"]:
    vals = df_raw[col].dropna().unique()
    if len(vals) > 2:
        issues.append({
            "Dimension": "Consistency",
            "Column":    col,
            "Issue":     f"Mixed casing: {sorted(vals)}",
            "Severity":  "High",
        })

# 4. Uniqueness — duplicate rows
dup_count = df_raw.duplicated().sum()
if dup_count > 0:
    issues.append({
        "Dimension": "Uniqueness",
        "Column":    "all",
        "Issue":     f"{dup_count} fully duplicate rows",
        "Severity":  "Medium",
    })

# 5. Accuracy — physically impossible values
impossible = {
    "MaxTemp":      (lambda s: s > 60),
    "MinTemp":      (lambda s: s < -30),
    "Pressure9am":  (lambda s: s < 800),
    "Pressure3pm":  (lambda s: s > 1100),
    "Humidity9am":  (lambda s: s > 100),
    "Humidity3pm":  (lambda s: s < 0),
    "Rainfall":     (lambda s: s > 400),
}
for col, cond in impossible.items():
    s = pd.to_numeric(df_raw[col], errors="coerce")
    flagged = int(cond(s).sum())
    if flagged > 0:
        issues.append({
            "Dimension": "Accuracy",
            "Column":    col,
            "Issue":     f"{flagged} physically impossible value(s)",
            "Severity":  "Critical",
        })

issues_df = pd.DataFrame(issues)
print("DATA QUALITY AUDIT REPORT")
print("=" * 75)
print(issues_df.to_string(index=False))

---
## 8. Data Curation — Outlier Detection

Three methods from the slides:
1. **Z-score**: $|z_i| > 3$; assumes normality
2. **IQR Rule**: $x < Q_1 - 1.5 \cdot \text{IQR}$ or $x > Q_3 + 1.5 \cdot \text{IQR}$; robust
3. **Isolation Forest**: algorithmic — anomalies isolated in fewer tree partitions

In [ ]:
# ── Build clean numeric frame for outlier detection ──────────────────────────
df_clean = df_raw.copy()
df_clean = df_clean.drop_duplicates()

# Fix string types
for col in ["Sunshine", "WindGustSpeed", "WindSpeed9am", "Evaporation"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Standardise categorical flags
for col in ["RainToday", "RainTomorrow"]:
    df_clean[col] = df_clean[col].str.capitalize()

NUM_COLS = ["MinTemp","MaxTemp","Rainfall","Evaporation","Sunshine",
            "WindGustSpeed","WindSpeed9am","WindSpeed3pm",
            "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm",
            "Cloud9am","Cloud3pm","Temp9am","Temp3pm","RISK_MM"]

# ── Method 1: Z-score ────────────────────────────────────────────────────────
print("METHOD 1 — Z-SCORE (|z| > 3)")
print("-" * 50)
zscore_flags = pd.DataFrame(False, index=df_clean.index, columns=NUM_COLS)
for col in NUM_COLS:
    s  = df_clean[col].dropna()
    mu, sigma = s.mean(), s.std()
    if sigma == 0:
        continue
    z = (df_clean[col] - mu) / sigma
    mask = z.abs() > 3
    zscore_flags[col] = mask
    n = int(mask.sum())
    if n > 0:
        vals = df_clean.loc[mask, col].tolist()
        print(f"  {col:20s}: {n:3d} outlier(s) → {vals[:5]}")

# ── Method 2: IQR Rule ───────────────────────────────────────────────────────
print("\nMETHOD 2 — IQR RULE (x < Q1−1.5·IQR or x > Q3+1.5·IQR)")
print("-" * 60)
iqr_flags = pd.DataFrame(False, index=df_clean.index, columns=NUM_COLS)
for col in NUM_COLS:
    s  = df_clean[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lo  = q1 - 1.5 * iqr
    hi  = q3 + 1.5 * iqr
    mask = (df_clean[col] < lo) | (df_clean[col] > hi)
    iqr_flags[col] = mask
    n = int(mask.sum())
    if n > 0:
        vals = df_clean.loc[mask, col].tolist()
        print(f"  {col:20s}: {n:3d} outlier(s)  bounds=[{lo:.1f}, {hi:.1f}]")

In [ ]:
from sklearn.ensemble import IsolationForest

# ── Method 3: Isolation Forest (multivariate) ────────────────────────────────
iso_cols  = ["MaxTemp", "MinTemp", "Pressure9am", "Humidity9am", "Rainfall"]
df_iso    = df_clean[iso_cols].copy()
df_iso    = df_iso.fillna(df_iso.median())

iso_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso_preds = iso_model.fit_predict(df_iso)   # -1 = anomaly, +1 = normal
iso_scores = iso_model.decision_function(df_iso)

anomaly_idx = np.where(iso_preds == -1)[0]
print(f"METHOD 3 — ISOLATION FOREST (contamination=5%)")
print(f"  Detected {len(anomaly_idx)} multivariate anomalies")
print("\nTop 10 anomalies (lowest anomaly score = most anomalous):")
sorted_idx = anomaly_idx[np.argsort(iso_scores[anomaly_idx])][:10]
print(df_clean.iloc[sorted_idx][iso_cols].to_string())

# ── Comparison: how many rows flagged by each method ─────────────────────────
print("\n--- OUTLIER DETECTION SUMMARY ---")
z_total   = int(zscore_flags.any(axis=1).sum())
iqr_total = int(iqr_flags.any(axis=1).sum())
iso_total = len(anomaly_idx)
print(f"  Z-score:          {z_total:3d} rows flagged")
print(f"  IQR:              {iqr_total:3d} rows flagged")
print(f"  Isolation Forest: {iso_total:3d} rows flagged")

In [ ]:
# ── Visual: IQR outlier flagging on MaxTemp and Pressure9am ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col in zip(axes, ["MaxTemp", "Pressure9am"]):
    s  = df_clean[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr     = q3 - q1
    lo, hi  = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    is_out  = (s < lo) | (s > hi)

    ax.scatter(range(len(s)), s.values,
               c=is_out.values.astype(int),
               cmap="bwr", alpha=0.6, s=15)
    ax.axhline(hi, color="red",   lw=1.5, linestyle="--", label=f"Q3+1.5·IQR = {hi:.1f}")
    ax.axhline(lo, color="blue",  lw=1.5, linestyle="--", label=f"Q1-1.5·IQR = {lo:.1f}")
    ax.set_title(f"{col} — {int(is_out.sum())} IQR outlier(s)", fontsize=10)
    ax.set_xlabel("Record index")
    ax.set_ylabel(col)
    ax.legend(fontsize=8)

plt.suptitle("IQR Outlier Detection — Red points are outside whiskers", fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Data Curation — Outlier Treatment

From the slides, treatment depends on context:
- **Remove**: if it's a confirmed data error
- **Cap / Winsorize**: clip extreme values to P5–P95 (good for models assuming normality)
- **Log / Box-Cox transform**: reduce right-skewness
- **Keep**: if the outlier encodes a legitimate real-world signal

> Tree-based models (Random Forest, XGBoost) are **naturally robust** to outliers — no treatment needed.

In [ ]:
# ── Strategy: remove impossible values, cap borderline outliers ──────────────
df_treated = df_clean.copy()

# 1. Remove physically impossible values (replace with NaN → imputed later)
impossible_bounds = {
    "MaxTemp":     (None, 55),
    "MinTemp":     (-20, None),
    "Pressure9am": (900, None),
    "Pressure3pm": (None, 1050),
    "Humidity9am": (0, 100),
    "Humidity3pm": (0, 100),
    "Rainfall":    (0, 300),
}
for col, (lo, hi) in impossible_bounds.items():
    if lo is not None:
        df_treated.loc[df_treated[col] < lo, col] = np.nan
    if hi is not None:
        df_treated.loc[df_treated[col] > hi, col] = np.nan

impossible_removed = sum(
    int(df_treated[c].isna().sum()) - int(df_clean[c].isna().sum())
    for c in impossible_bounds
)
print(f"Impossible values replaced with NaN: {impossible_removed}")

# 2. Winsorize remaining numeric columns at P5–P95
cap_cols = ["WindGustSpeed", "WindSpeed9am", "WindSpeed3pm", "Rainfall"]
print("\nWinsorization (P5–P95):")
for col in cap_cols:
    p5  = df_treated[col].quantile(0.05)
    p95 = df_treated[col].quantile(0.95)
    before = df_treated[col].describe()[["min","max"]].to_dict()
    df_treated[col] = df_treated[col].clip(lower=p5, upper=p95)
    after  = df_treated[col].describe()[["min","max"]].to_dict()
    print(f"  {col:20s}: min {before['min']:.1f}→{after['min']:.1f}  "
          f"max {before['max']:.1f}→{after['max']:.1f}")

# 3. Log1p transform for Rainfall (highly right-skewed)
df_treated["Rainfall_log1p"] = np.log1p(df_treated["Rainfall"])
print(f"\nRainfall log1p transform: skew {df_treated['Rainfall'].skew():.2f} "
      f"→ {df_treated['Rainfall_log1p'].skew():.2f}")

# ── Before / After distribution ───────────────────────────────────────────────
# For the log1p panel we apply log1p to the "before" series too so that
# both curves live on the same axis (log-mm scale).  Otherwise raw Rainfall
# (0–40 mm) and log1p(Rainfall) (0–2.2) would be plotted together, making
# the "after" curve look like a narrow spike / outlier near x = 0.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_spec = [
    ("MaxTemp",  df_clean["MaxTemp"],                   "MaxTemp",        "MaxTemp (impossible→NaN)"),
    ("Rainfall", df_clean["Rainfall"],                  "Rainfall",       "Rainfall (winsorized)"),
    ("log1p(Rainfall)", np.log1p(df_clean["Rainfall"]), "Rainfall_log1p", "Rainfall (log1p transform)"),
]
for ax, (xlabel, before_s, trt_c, label) in zip(axes, plot_spec):
    before_s.dropna().plot.kde(ax=ax, color="tomato",    lw=2, label="before")
    df_treated[trt_c].dropna().plot.kde(ax=ax, color="steelblue", lw=2, label="after")
    ax.set_title(label, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.legend(fontsize=8)

plt.suptitle("Outlier Treatment — Distribution Comparison", fontsize=12)
plt.tight_layout()
plt.show()


---
## 10. Data Curation — Missing Data & Imputation

From the slides, missing-data mechanisms:
| Mechanism | Meaning | Example |
|-----------|---------|---------|
| **MCAR** | Missing completely at random — no pattern | Sensor randomly dropped reading |
| **MAR** | Missing at random — depends on other observed variables | Sunshine missing only on rainy days |
| **MNAR** | Missing not at random — depends on the unobserved value itself | Humidity not recorded when extremely high |

Imputation strategies:
- **Mean / Median**: simple; use mean for symmetric, median for skewed distributions
- **KNN Imputer**: uses K-nearest neighbours based on feature similarity
- **Iterative Imputer (MICE)**: models each feature as function of others

In [ ]:
from sklearn.impute import KNNImputer, SimpleImputer

# ── Missing data analysis ─────────────────────────────────────────────────────
null_counts = df_treated[NUM_COLS].isnull().sum()
null_pct    = null_counts / len(df_treated) * 100

print("Missing values after outlier treatment:")
print(null_counts[null_counts > 0].to_frame("count").assign(pct=null_pct).to_string())

# Classify mechanisms (expert knowledge / domain assumption)
mechanisms = {
    "Evaporation":  "MCAR",
    "Sunshine":     "MAR  (likely correlated with Cloud cover)",
    "Cloud9am":     "MAR  (weather-event driven)",
    "Cloud3pm":     "MAR  (weather-event driven)",
    "Humidity3pm":  "MAR  (may correlate with RainToday)",
    "MaxTemp":      "MNAR (extreme values removed → systematic gap)",
    "Pressure9am":  "MNAR (extreme values removed → systematic gap)",
}
print("\nEstimated missing-data mechanisms:")
for col, mech in mechanisms.items():
    if null_counts.get(col, 0) > 0:
        print(f"  {col:20s}: {mech}")

# ── Imputation: median for MCAR/MNAR columns, KNN for MAR ────────────────────
df_imputed = df_treated.copy()

# Median imputation for selected columns
median_imp_cols = ["MaxTemp", "MinTemp", "Pressure9am", "Pressure3pm",
                   "Evaporation", "Humidity9am", "Humidity3pm"]
for col in median_imp_cols:
    med = df_imputed[col].median()
    n_filled = int(df_imputed[col].isna().sum())
    df_imputed[col] = df_imputed[col].fillna(med)
    if n_filled > 0:
        print(f"  Median imputation: {col:20s} filled {n_filled} values with {med:.2f}")

# KNN imputation for correlated columns
knn_cols = ["Sunshine", "Cloud9am", "Cloud3pm"]
knn_data = df_imputed[knn_cols + ["Humidity3pm", "Rainfall"]].copy()
knn_imp  = KNNImputer(n_neighbors=5)
knn_filled = knn_imp.fit_transform(knn_data)
for i, col in enumerate(knn_cols):
    n_filled = int(df_treated[col].isna().sum())
    df_imputed[col] = knn_filled[:, i]
    if n_filled > 0:
        print(f"  KNN  imputation:   {col:20s} filled {n_filled} values")

print("\nNull counts after imputation:")
remaining = df_imputed[NUM_COLS].isnull().sum()
print(remaining[remaining > 0].to_string() if remaining.any() else "  → No nulls remaining in numeric columns ✓")

---
## 11. Data Curation — Feature Engineering

Techniques from the slides:
- **Categorical encoding**: ordinal map → binary; one-hot for nominal
- **Feature scaling**: StandardScaler (zero mean, unit variance)
- **Derived features**: domain knowledge (TempRange = MaxTemp − MinTemp)
- **Log transform**: reduce right-skewness

In [ ]:
from sklearn.preprocessing import StandardScaler

df_feat = df_imputed.copy()

# 1. Binary encode RainToday & RainTomorrow (already standardised to Title case)
for col in ["RainToday", "RainTomorrow"]:
    df_feat[col] = df_feat[col].map({"Yes": 1, "No": 0})
print("Binary encoding:")
print(df_feat[["RainToday", "RainTomorrow"]].value_counts().to_string())

# 2. Derived feature: temperature range
df_feat["TempRange"] = df_feat["MaxTemp"] - df_feat["MinTemp"]
print(f"\nTempRange stats:\n{df_feat['TempRange'].describe().to_string()}")

# 3. Log1p transform for Rainfall (already added, ensure consistent)
df_feat["Rainfall_log1p"] = np.log1p(df_feat["Rainfall"])

# 4. One-hot encode WindGustDir
wind_dummies = pd.get_dummies(df_feat["WindGustDir"], prefix="WGD", drop_first=True)
df_feat = pd.concat([df_feat, wind_dummies], axis=1)
df_feat.drop(columns=["WindGustDir","WindDir9am","WindDir3pm"], inplace=True, errors="ignore")
print(f"\nOne-hot encoded WindGustDir → {len(wind_dummies.columns)} new binary columns")

# 5. StandardScaler on numeric features
scale_cols  = [c for c in NUM_COLS if c in df_feat.columns]
scaler      = StandardScaler()
df_scaled   = df_feat.copy()
df_scaled[scale_cols] = scaler.fit_transform(df_feat[scale_cols])

print("\nAfter StandardScaler (sample means ≈ 0, std ≈ 1):")
print(df_scaled[scale_cols[:6]].agg(["mean","std"]).round(3).to_string())
print(f"\nFinal feature matrix shape: {df_scaled.shape}")

---
## 12. Data Curation — Feature Selection

From the slides, three main families:
- **Filter**: correlation matrix, mutual information — independent of model
- **Wrapper**: RFE, forward/backward search — slower but model-aware
- **Embedded**: LASSO (L1), Random Forest importance — selection during training

Rule of thumb: drop features with pairwise $|r| > 0.9$ (multicollinearity), then rank remaining by model importance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ── Filter method: remove highly correlated features (|r| > 0.9) ─────────────
corr_num = df_feat[NUM_COLS].corr().abs()
upper    = corr_num.where(np.triu(np.ones_like(corr_num, dtype=bool), k=1))
drop_filter = [col for col in upper.columns if any(upper[col] > 0.9)]
print("Filter — highly correlated features to drop (|r| > 0.9):")
for col in drop_filter:
    corr_with = upper[col][upper[col] > 0.9].index.tolist()
    print(f"  Drop '{col}' (correlated with {corr_with})")

# ── Embedded: Random Forest feature importances ───────────────────────────────
target = "RainTomorrow"
feat_cols = [c for c in NUM_COLS + ["TempRange"] if c in df_feat.columns and c != target]

X = df_feat[feat_cols].fillna(df_feat[feat_cols].median())
y = df_feat[target].dropna()
X = X.loc[y.index]

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("\nRandom Forest — Top 12 feature importances:")
print(importances.head(12).round(4).to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
importances.head(12).sort_values().plot.barh(ax=ax, color="#4c72b0", edgecolor="white")
ax.set_xlabel("Importance score")
ax.set_title("Random Forest Feature Importances (Top 12)\nfor predicting RainTomorrow", fontsize=11)
plt.tight_layout()
plt.show()

---
## Summary — EDA & Data Curation Pipeline

| Step | Action | Key Finding |
|------|--------|-------------|
| **1 – Load** | Import `weather_problems.json` (372 rows × 22 cols) | 6 duplicate rows + 8 injected outliers |
| **2 – Profile** | `df.info()`, null counts, dtype check | `Evaporation` stored as string; 5 cols with > 7 % nulls |
| **3 – Univariate** | Histograms, KDE, box plots | Rainfall strongly right-skewed; pressure has extreme outliers |
| **4 – Descriptive stats** | Mean, median, std, IQR, range | Mean–median gap in Rainfall → skewness confirmed |
| **5 – Shape** | Q1/Q3, P95, skewness, kurtosis | Rainfall: skew > 10 (leptokurtic); MaxTemp ≈ normal |
| **6 – Bivariate** | Pearson + Spearman heatmaps, scatter plots | Temp9am–Temp3pm r = 0.99; Pressure9am–Pressure3pm r = 0.98 |
| **7 – Quality audit** | 7 quality dimensions | Critical: 8 accuracy violations; High: type mismatches, mixed casing |
| **8 – Outlier detection** | Z-score, IQR, Isolation Forest | All three methods detect the 8 injected outliers |
| **9 – Outlier treatment** | Remove impossible, winsorize P5–P95, log1p | Rainfall distribution normalised after log transform |
| **10 – Imputation** | Median (MCAR/MNAR cols), KNN (MAR cols) | Zero nulls remaining in numeric columns after imputation |
| **11 – Feature engineering** | Binary encode, one-hot WindGustDir, StandardScaler, TempRange | Final matrix: ~371 rows × 35+ features |
| **12 – Feature selection** | Correlation filter \|r\|>0.9, RF importances | Top predictors: Humidity3pm, Pressure3pm, RISK_MM, Temp3pm |

> **Next step**: apply a classification model (Logistic Regression, Random Forest, XGBoost) on the curated dataset to predict `RainTomorrow`.